# Dog vs Cat Image Classification
### Fine-tuning ResNet18 using PyTorch
**Dataset:** [Kaggle - Dog and Cat Classification Dataset](https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset)  
**Model:** ResNet18 (pretrained on ImageNet, fine-tuned)  
**Task:** Binary Image Classification (Dog vs Cat)

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision flask numpy Pillow scikit-learn

## 2. Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 3. Configuration

In [ ]:
DATASET_DIR  = "./dataset"
TRAIN_DIR    = os.path.join(DATASET_DIR, "training_set")
TEST_DIR     = os.path.join(DATASET_DIR, "test_set")
OUTPUT_DIR   = "./model_output"
MODEL_PATH   = os.path.join(OUTPUT_DIR, "resnet18_dogcat.pth")

IMAGE_SIZE   = 64
BATCH_SIZE   = 64
EPOCHS       = 2
LEARNING_RATE = 1e-4
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ID2LABEL     = {0: "cat", 1: "dog"}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration set!")

## 4. Data Transforms

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print("Transforms defined!")

## 5. Load Dataset

In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(TEST_DIR,  transform=val_transforms)

# Use subset for faster training
train_dataset = Subset(train_dataset, range(2000))
val_dataset   = Subset(val_dataset,   range(500))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Classes       : {train_dataset.dataset.classes}")
print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")

## 6. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Sample Images from Dataset', fontsize=14)

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i, ax in enumerate(axes.flatten()):
    img, label = train_dataset[i]
    img = img.numpy().transpose(1, 2, 0)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(ID2LABEL[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Build Model (ResNet18)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 2)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

print("Model built successfully!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 8. Train the Model

In [ ]:
train_accs, val_accs, train_losses, val_losses = [], [], [], []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    for phase, loader in [("TRAIN", train_loader), ("VAL", val_loader)]:
        model.train() if phase == "TRAIN" else model.eval()
        running_loss, all_preds, all_labels = 0, [], []

        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == "TRAIN"):
                outputs = model(inputs)
                loss    = criterion(outputs, labels)
                preds   = torch.argmax(outputs, dim=1)
                if phase == "TRAIN":
                    loss.backward()
                    optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = accuracy_score(all_labels, all_preds)
        print(f"  [{phase}] Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

        if phase == "TRAIN":
            train_losses.append(epoch_loss)
            train_accs.append(epoch_acc)
        else:
            val_losses.append(epoch_loss)
            val_accs.append(epoch_acc)

print("\n✅ Training complete!")

## 9. Plot Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_accs, label='Train Accuracy', marker='o')
ax1.plot(val_accs,   label='Val Accuracy',   marker='o')
ax1.set_title('Accuracy per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(train_losses, label='Train Loss', marker='o')
ax2.plot(val_losses,   label='Val Loss',   marker='o')
ax2.set_title('Loss per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 10. Save Model

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names":      train_dataset.dataset.classes,
    "id2label":         ID2LABEL,
    "image_size":       IMAGE_SIZE,
}, MODEL_PATH)

print(f"✅ Model saved to: {MODEL_PATH}")

## 11. Batch Inference on 4 Samples

In [ ]:
SAMPLES = [
    {"image_path": "./dataset/test_set/dogs/9977.jpg", "true_label": 1},
    {"image_path": "./dataset/test_set/cats/9977.jpg", "true_label": 0},
    {"image_path": "./dataset/test_set/dogs/9978.jpg", "true_label": 1},
    {"image_path": "./dataset/test_set/cats/9978.jpg", "true_label": 0},
]

infer_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

model.eval()
true_labels, pred_labels = [], []

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Batch Inference Results', fontsize=14)

for i, sample in enumerate(SAMPLES):
    img       = Image.open(sample["image_path"]).convert("RGB")
    tensor    = infer_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pred      = int(np.argmax(probs))
    true      = sample["true_label"]
    true_labels.append(true)
    pred_labels.append(pred)

    color = "green" if pred == true else "red"
    axes[i].imshow(img)
    axes[i].set_title(f"Pred: {ID2LABEL[pred]}\nTrue: {ID2LABEL[true]}\n{max(probs)*100:.1f}%", color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 12. Evaluation Metrics

In [ ]:
acc = accuracy_score(true_labels, pred_labels)
f1  = f1_score(true_labels, pred_labels, average="weighted", zero_division=0)

print("📊 EVALUATION METRICS")
print(f"   Accuracy : {acc:.4f} ({acc*100:.1f}%)")
print(f"   F1 Score : {f1:.4f}")
print()
print(classification_report(true_labels, pred_labels, target_names=["Cat", "Dog"], zero_division=0))

## 13. Summary

| Item | Detail |
|---|---|
| **Model** | ResNet18 (pretrained on ImageNet) |
| **Task** | Binary Image Classification |
| **Dataset** | Dog and Cat Classification (Kaggle) |
| **Train Samples** | 2000 |
| **Val Samples** | 500 |
| **Epochs** | 2 |
| **Optimizer** | Adam (lr=0.0001) |
| **Evaluation Metric** | Accuracy + Weighted F1 Score |
| **Docker Hub** | https://hub.docker.com/r/jabbar316/dogcat-api |
| **GitHub** | https://github.com/Hazzyness/dogcat-classifier |